# 06 — Groupby y agregaciones

`groupby` es la operación más importante de pandas para análisis de datos. Divide el DataFrame en grupos, aplica una función a cada grupo, y combina los resultados.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
print(df.shape)


(9800, 18)


## Groupby básico

In [2]:
# groupby devuelve un objeto GroupBy — no calcula nada hasta que se agrega
g = df.groupby('Region')
print(type(g))
print('Grupos:', list(g.groups.keys()))
print()

# Aplicar una función de agregación
print(g['Sales'].sum())
print()
print(g['Sales'].mean().round(2))


<class 'pandas.api.typing.DataFrameGroupBy'>
Grupos: ['Central', 'East', 'South', 'West']

Region
Central    492646.9132
East       669518.7260
South      389151.4590
West       710219.6845
Name: Sales, dtype: float64

Region
Central    216.36
East       240.40
South      243.52
West       226.18
Name: Sales, dtype: float64


## agg() con lista de funciones

In [3]:
# Aplicar múltiples funciones a la misma columna
resultado = (
    df.groupby('Category')['Sales']
    .agg(['sum', 'mean', 'count', 'min', 'max'])
    .round(2)
)
print(resultado)

# El resultado tiene MultiIndex en columnas si se agregan varias columnas


                       sum    mean  count   min       max
Category                                                 
Furniture        728658.58  350.65   2078  1.89   4416.17
Office Supplies  705422.33  119.38   5909  0.44   9892.74
Technology       827455.87  456.40   1813  0.99  22638.48


## agg() con agregaciones nombradas (tuplas)

La forma más profesional — produce columnas con nombres exactos sin necesidad de renombrar después.

In [4]:
resultado = (
    df.groupby('Region')['Sales']
    .agg(
        revenue_total = 'sum',
        ticket_medio  = 'mean',
        num_pedidos   = 'count',
        venta_max     = 'max',
    )
    .round(2)
    .sort_values('revenue_total', ascending=False)
    .reset_index()
)
print(resultado)


    Region  revenue_total  ticket_medio  num_pedidos  venta_max
0     West      710219.68        226.18         3140   13999.96
1     East      669518.73        240.40         2785   11199.97
2  Central      492646.91        216.36         2277   17499.95
3    South      389151.46        243.52         1598   22638.48


## agg() con diccionario — distintas funciones por columna

In [5]:
# Cuando se quieren funciones distintas para columnas distintas
resultado = df.groupby('Category').agg({
    'Sales':       ['sum', 'mean'],
    'Order ID':    'nunique',
    'Customer ID': 'nunique',
})

# El resultado tiene MultiIndex en columnas — aplanarlo
resultado.columns = ['_'.join(col) for col in resultado.columns]
resultado = resultado.reset_index()
print(resultado)


          Category    Sales_sum  Sales_mean  Order ID_nunique  \
0        Furniture  728658.5757  350.653790              1727   
1  Office Supplies  705422.3340  119.381001              3676   
2       Technology  827455.8730  456.401474              1519   

   Customer ID_nunique  
0                  705  
1                  787  
2                  684  


## Groupby por múltiples columnas

In [6]:
resultado = (
    df.groupby(['Region', 'Category'])['Sales']
    .agg(revenue='sum', pedidos='count')
    .reset_index()
    .sort_values(['Region', 'revenue'], ascending=[True, False])
)
print(resultado)


     Region         Category      revenue  pedidos
2   Central       Technology  168739.2080      408
1   Central  Office Supplies  163590.2430     1399
0   Central        Furniture  160317.4622      470
5      East       Technology  263116.5270      527
3      East        Furniture  206461.3880      591
4      East  Office Supplies  199940.8110     1667
8     South       Technology  148195.2080      289
7     South  Office Supplies  124424.7710      983
6     South        Furniture  116531.4800      326
11     West       Technology  247404.9300      589
9      West        Furniture  245348.2455      691
10     West  Office Supplies  217466.5090     1860


## transform() — devuelve una Serie del mismo tamaño que el input

A diferencia de `agg()`, que reduce el número de filas, `transform()` devuelve un valor por cada fila del grupo. Útil para crear columnas basadas en estadísticas de grupo.

In [7]:
# Añadir la media del grupo como columna nueva (sin reducir filas)
df['media_region'] = df.groupby('Region')['Sales'].transform('mean').round(2)

# Diferencia de cada venta respecto a la media de su región
df['desviacion_media'] = (df['Sales'] - df['media_region']).round(2)

print(df[['Region', 'Sales', 'media_region', 'desviacion_media']].head(10))


  Region     Sales  media_region  desviacion_media
0  South  261.9600        243.52             18.44
1  South  731.9400        243.52            488.42
2   West   14.6200        226.18           -211.56
3  South  957.5775        243.52            714.06
4  South   22.3680        243.52           -221.15
5   West   48.8600        226.18           -177.32
6   West    7.2800        226.18           -218.90
7   West  907.1520        226.18            680.97
8   West   18.5040        226.18           -207.68
9   West  114.9000        226.18           -111.28


## pivot_table

Crea tablas cruzadas con filas, columnas, y valores. Más fácil de leer que un groupby multidimensional.

In [8]:
tabla = pd.pivot_table(
    df,
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc='sum',
    margins=True,        # añade fila/columna de totales
    margins_name='Total'
).round(0)

print(tabla)


Category  Furniture  Office Supplies  Technology      Total
Region                                                     
Central    160317.0         163590.0    168739.0   492647.0
East       206461.0         199941.0    263117.0   669519.0
South      116531.0         124425.0    148195.0   389151.0
West       245348.0         217467.0    247405.0   710220.0
Total      728659.0         705422.0    827456.0  2261537.0


---
## Resumen

| Patrón | Sintaxis |
|--------|----------|
| Suma por grupo | `df.groupby('col')['val'].sum()` |
| Múltiples funciones | `.agg(['sum', 'mean', 'count'])` |
| Nombres exactos | `.agg(total='sum', media='mean')` |
| Distintas funciones por columna | `.agg({'col1': 'sum', 'col2': 'nunique'})` |
| Aplanar MultiIndex | `df.columns = ['_'.join(c) for c in df.columns]` |
| Sin reducir filas | `.transform('mean')` |
| Tabla cruzada | `pd.pivot_table(df, values, index, columns, aggfunc)` |
